# 01. Kimi K3 기초 계산

## 목표
MoE 라우팅 비율과 긴 문맥의 단순 비용 모델을 계산하며 핵심 용어를 이해합니다. Python 3 표준 라이브러리만 사용합니다.

In [ ]:
total_experts = 896
active_experts = 16
routing_ratio = active_experts / total_experts
print(f"전문가 개수 기준 활성 비율: {routing_ratio:.2%}")
# 이 값은 공유 계층을 포함한 실제 활성 파라미터 비율이 아닙니다.
# 전문가별 크기와 공유 파라미터가 공개돼야 정확히 계산할 수 있습니다.

## 긴 문맥 비용의 직관
표준 어텐션의 점수 행렬 크기는 시퀀스 길이의 제곱에 비례합니다. 아래 계산은 실제 KDA 구현 비용이 아니라, 왜 효율적 긴 문맥 기법이 필요한지 보여주는 비교용 모델입니다.

In [ ]:
lengths = [1_000, 10_000, 100_000, 1_000_000]
baseline = lengths[0] ** 2
for length in lengths:
    relative = length ** 2 / baseline
    print(f"{length:>9,} tokens -> 1천 토큰 대비 {relative:>12,.0f}배")
# 토큰 수가 10배면 단순 점수 행렬 원소 수는 100배가 됩니다.

## 공개 가중치와 활성 파라미터
2026-07-28 Hugging Face 모델 카드는 전체 2.8T, 활성 104B 파라미터와 약 1.56TB 저장소를 명시합니다. 아래 계산으로 전체 규모와 한 토큰에서 활성화되는 규모를 구분합니다.

In [ ]:
total_parameters = 2.8e12
active_parameters = 104e9
repository_size_tb = 1.56

active_ratio = active_parameters / total_parameters
print(f"파라미터 기준 활성 비율: {active_ratio:.2%}")
print(f"가중치 3벌 보관 예상: {repository_size_tb * 3:.2f} TB")

# expert 개수 기준 16/896과 파라미터 기준 104B/2.8T는 다릅니다.
# shared layer, vision encoder, expert별 구조가 전체 계산에 포함되기 때문입니다.

## 연습
1. 활성 전문가 수를 8, 32로 바꾸어 비율을 비교하세요.
2. 시퀀스 길이에 선형인 가상 기법과 제곱 비용을 비교하세요.
3. 다운로드 중 임시 복사본과 변환본까지 고려해 1.56TB 모델에 필요한 여유 저장 공간을 계산하세요.
4. '최대 문맥 길이'와 '정보 회수 품질'이 다른 이유를 한 문장으로 적어보세요.